### Google Colaboratory setup

In [1]:
#@title 0. SQAI handson environment setup

from pathlib import Path
import os
import subprocess
import sys


def running_on_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if running_on_colab():
    REPO_URL = "https://github.com/tiksato/SQAI_handson.git"
    REPO_REF = "main"
    PROJECT_ROOT = Path("/content/SQAI_handson")

    if not PROJECT_ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--quiet",
                "--depth", "1",
                "--branch", REPO_REF,
                REPO_URL,
                str(PROJECT_ROOT),
            ],
            check=True,
        )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--editable",
            str(PROJECT_ROOT),
        ],
        check=True,
    )

    os.chdir(PROJECT_ROOT)

else:
    # Local case
    current = Path.cwd().resolve()

    if current.name == "notebooks":
        PROJECT_ROOT = current.parent
    else:
        PROJECT_ROOT = current

print(
    "Environment:",
    "Google Colab" if running_on_colab() else "Local Python",
)
print("Project root:", PROJECT_ROOT)

Environment: Local Python
Project root: /Users/sato/project/pyqmd_work/SQAI_handson/SQAI_notebooks


In [2]:
import time
import math
import numpy as np
from pyscf import gto, scf

from SQAI_modules.misc_utils.matrix_utilities import davidson
from SQAI_modules.ormas_tools.rasci import Full_CI, RAS_CI, RHF_CI

In [3]:
basis_list = [
    'STO-3G', 
    '6-31G', 
    '6-31G(d,p)',
    '6-31G(df,pd)'
]

def calc_mo_integrals(mol):
    # Hamiltonian matrix elements generation                                                         
    print("Running RHF to obtain MO integrals...")
    t0 = time.time()
    HF = scf.RHF(mol)
    HF.verbose = 0
    HF.run()
    E_HF = HF.e_tot
    print(f"HF Energy:   {E_HF:.10f} Hartree")

    # Nuclear repulsion energy
    H_const = mol.energy_nuc()

    # AO integrals
    T = mol.intor("int1e_kin_sph") # AO kinetic
    Vnuc = mol.intor("int1e_nuc_sph") # AO Nuc-elec
    g_AO = mol.intor("int2e_sph").transpose(0,2,1,3) # AO ERIs in physics format

    # MO transformation
    CMO = HF.mo_coeff
    h_AO = T + Vnuc
    h_MO = np.einsum("pi,qj,pq->ij", CMO, CMO, h_AO, 
                     optimize=True)
    g_MO = np.einsum("pi,qj,rk,sl,pqrs->ijkl", CMO, CMO, CMO, CMO, g_AO, 
                     optimize=True)
    
    print(f" ... Done: {time.time()-t0:.2f} sec.")

    return H_const, h_MO, g_MO

def calc_rasci(R_HH, basis):

    print("##################################")
    #%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
    # Mol object generation                                                                         
    mol = gto.M(
        atom=f'''
        H {-5*R_HH/2} 0 0
        H {-3*R_HH/2} 0 0
        H   {-R_HH/2} 0 0
        H    {R_HH/2} 0 0
        H  {3*R_HH/2} 0 0
        H  {5*R_HH/2} 0 0
        ''',
        basis=basis, 
        symmetry=False, 
        verbose=0,
    )    
    n_act = mol.nao # number of active orbitals
    n_elec = np.array(mol.nelec) # active electrons, each spin                                                      
    print(f"basis = {basis}: n_act  = {n_act}")


    #%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
    # MO integrals generation                                                                         
    H_const, h_MO, g_MO = calc_mo_integrals(mol)


    #%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
    # CI instance generation                                                                         
    print("Constructing CI object...")
    t0 = time.time()
    n_RAS1 = 0
    n_RAS2 = sum(n_elec)
    n_RAS3 = n_act - n_RAS1 - n_RAS2
    myCI = RAS_CI(
        n_elec = n_elec,
        n_orb_RAS = (n_RAS1, n_RAS2, n_RAS3), 
        max_hole_RAS1 = 0, 
        max_elec_RAS3 = 2,
        num_threads = 10
    )
    print(f" ... Done: {time.time()-t0:.2f} sec.")
    print(f"CI dimension: {myCI.total_dim}")
    print("String distribution over occupation groups:")
    print(myCI.mat_num_str)


    #%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
    # Direct-CI Davidson diagonalization
    print("Davidson diagonalization...")
    t0 = time.time()
    def my_get_sigma(x):
        return myCI.h_prod(h_MO, g_MO, x).real

    hdiag = myCI.calc_hdiag(h_MO, g_MO).real
    def my_precond(dx, e, x0, small=1e-3):
        denom = hdiag - e
        return dx * denom / (denom**2 + small)

    Ene, CIC = davidson(myCI.total_dim,
                        my_get_sigma,
                        my_precond,
                        tol = 1E-6,
                        verbose=5)
    print(f" ... Done: {time.time()-t0:.2f} sec.")
    print(f"CI energy: {Ene + H_const}")    
    print()



for basis in basis_list:
    calc_rasci(R_HH = 1.0, basis=basis)

##################################
basis = STO-3G: n_act  = 6
Running RHF to obtain MO integrals...
HF Energy:   -3.1355322140 Hartree
 ... Done: 0.01 sec.
Constructing CI object...
 ... Done: 5.04 sec.
CI dimension: 400
String distribution over occupation groups:
[[400]]
Davidson diagonalization...
verbose = 5
davidson 0 1  |r|= 0.349  e= [-7.73937395]  max|de|= -7.74  lindep=    1
davidson 1 2  |r|= 0.136  e= [-7.83020459]  max|de|= -0.0908  lindep= 0.999
davidson 2 3  |r|= 0.0471  e= [-7.83882271]  max|de|= -0.00862  lindep= 0.966
davidson 3 4  |r|= 0.0136  e= [-7.83978825]  max|de|= -0.000966  lindep= 0.979
davidson 4 5  |r|= 0.00681  e= [-7.83987738]  max|de|= -8.91e-05  lindep= 0.971
davidson 5 6  |r|= 0.00341  e= [-7.83990278]  max|de|= -2.54e-05  lindep= 0.942
davidson 6 7  |r|= 0.00087  e= [-7.83990773]  max|de|= -4.95e-06  lindep= 0.977
root 0 converged  |r|= 0.000205  e= -7.839907987552873  max|de|= -2.58e-07
converged 7 8  |r|= 0.000205  e= [-7.83990799]  max|de|= -2.58e-07